# The dataset is public now. Enjoy!⭐⭐⭐

For each task this notebook takes the cheapest circuit we know of, re-verifies every file here with the
unmodified official verifiers, and writes `/kaggle/working/submission.zip`.

| Task | Circuit | Ratio |
|---|---|---:|
| invsqrt | yoyobar's 62-qubit file | 20.625 |
| DQI | yoyobar's 888-qubit decoder | 61.337 |
| dfthcsos | Atakan Aldemir's learned angle words on SomeJoint's R = 8, B = 143, C = 60 factorization, table weights ours (from Ahmed Berat Özer's, manwithacat's, Dmitrii Gluzdov's, Sargis Khachatryan's, sigmaborov's and baidalinadilzhan's), angle circuits as assembled by Alex Chilton (five of them ours) | 9.941 |

## Re-verify every component here

Each file is copied from the release dataset and matched to its SHA-256; then the three official verifiers from the competition data run on it, unmodified, at their default shot counts.

In [ ]:
import glob, hashlib, json, lzma, math, os, shutil, sys, time, zipfile, pathlib

def find_comp():
    for p in [os.environ.get('QCC_COMP_DIR', ''), '/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation',
              '/kaggle/input/google-verifiable-quantum-circuit-compilation']:
        if p and os.path.isdir(os.path.join(p, 'verifiers')):
            return p
    raise SystemExit('competition data not found')

def find_release():
    if os.environ.get('QCC_DS_DIR'):
        return os.environ['QCC_DS_DIR']
    hits = glob.glob('/kaggle/input/**/QCC_RELEASE_MANIFEST.json', recursive=True)
    if not hits:
        raise SystemExit('release dataset not found')
    return os.path.dirname(hits[0])

COMP, DS = find_comp(), find_release()
WORK = pathlib.Path(os.environ.get('QCC_WORK_DIR', '/kaggle/working'))
SUB = WORK / 'submission'
sys.path[:0] = [os.path.join(COMP, 'kickmix'), os.path.join(COMP, 'verifiers')]
import kickmix as km
from invsqrt import verify_submission_invsqrt as vinv
from dqi import verify_submission_dqi as vdqi
from dfthcsos import verify as vdft
MANIFEST = json.load(open(os.path.join(DS, 'QCC_RELEASE_MANIFEST.json')))
EXPECTED_PORTFOLIO = 'v56_yb'
assert MANIFEST['portfolio'] == EXPECTED_PORTFOLIO, ('stale dataset version', MANIFEST['portfolio'], EXPECTED_PORTFOLIO)
SALT = MANIFEST['salt']
print('competition data:', COMP, '| release dataset:', DS, '| salt:', SALT)

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

if SUB.exists():
    shutil.rmtree(SUB)
for item in MANIFEST['files']:
    src, dst = os.path.join(DS, item['stored_as']), SUB / item['file']
    dst.parent.mkdir(parents=True, exist_ok=True)
    if item['stored_as'].endswith('.xz'):
        with lzma.open(src) as fi, open(dst, 'wb') as fo:
            shutil.copyfileobj(fi, fo, 1 << 22)
    else:
        shutil.copyfile(src, dst)
    got = sha256(dst)
    assert got == item['sha256'], (item['file'], got)
    print('sha256', item['file'], got)
print(len(MANIFEST['files']), 'files copied and matched to the manifest by SHA-256')

In [ ]:
REF = MANIFEST['reference_costs']
results = {}
t0 = time.time()
circ = km.Circuit.from_file(str(SUB / 'invsqrt.kmb'), format='kmb')
results['invsqrt'] = vinv.verify(circ, vinv.ProblemSpecInvSqrt.for_n_p(10), salt=SALT)
print('invsqrt done', round(time.time() - t0), 's'); t0 = time.time()
circ = km.Circuit.from_file(str(SUB / 'dqi.kmb'), format='kmb')
results['dqi'] = vdqi.verify(circ, vdqi.ProblemSpecDQI.benchmark_instance(), salt=SALT)
del circ
print('dqi done', round(time.time() - t0), 's'); t0 = time.time()
results['dfthcsos'] = vdft.verify(str(SUB / 'dfthcsos'), salt=SALT)
print('dfthcsos done', round(time.time() - t0), 's')
rows, ratios = [], []
for task in ('invsqrt', 'dqi', 'dfthcsos'):
    r = results[task]
    assert r['status'] == 'PASS', (task, r)
    ratio = REF[task] / float(r['score'])
    ratios.append(ratio)
    if task == 'invsqrt':
        extra = f"{r['num_qubits']} qubits, {r['toffoli_equivalent']:.1f} Toffoli-equivalents"
    elif task == 'dqi':
        extra = (f"{r['num_qubits']} qubits, {r['toffoli_count']:.0f} mean Toffolis, "
                 f"{r.get('value_errors', 0)} value / {r.get('phase_errors', 0)} phase errors")
    else:
        extra = f"lambda_eff {r['lambda_eff']:.3f}, {r['toffoli_count']:.0f} Toffolis, {r['num_qubits']} qubits"
    rows.append((task, float(r['score']), ratio, extra))
    print(f"{task:9s} PASS cost {float(r['score']):.6g}  ratio {ratio:8.4f}  ({extra})")
gm = math.prod(ratios) ** (1 / 3)
print(f'geometric mean of reference / cost: {gm:.4f}')

## Write the submission

In [ ]:
zip_path = WORK / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for item in MANIFEST['files']:
        z.write(SUB / item['file'], item['file'])
shutil.rmtree(SUB)
print('wrote', zip_path, round(zip_path.stat().st_size / 2**20, 1), 'MiB')

# Sources and credits

- **invsqrt** — yoyobar's 62-qubit file, byte for byte, from the output of
  https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits (Apache-2.0; dataset
  https://www.kaggle.com/datasets/yoyobar/qcc-local-sota-20261031, upstream Apache-2.0 notice included there).
  Lineage as credited there: yoyobar's own 62-qubit work (developed with GPT-6 Astra) on Akmal Xodarev's (busyaprime)
  https://www.kaggle.com/code/busyaprime/lb-22-86841-22x-cheaper-than-google-s-circuits and mrmeows / team selenium
  (https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits), with Ahmed Berat Özer's patch
  (https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression) as distributed in manwithacat's
  https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo; earlier lineage: Phuc Thai (DePie), Dmitrii Gluzdov,
  marsyas.

- **dqi** — yoyobar's 888-qubit decoder, byte for byte, from the output of
  https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits (Apache-2.0; dataset
  https://www.kaggle.com/datasets/yoyobar/qcc-local-sota-20261031, upstream Apache-2.0 notice included there).
  Earlier public decoders in this family, as credited in our previous versions: Agniv Banerjee (SomeJoint)'s
  889-qubit decoders (https://www.kaggle.com/code/somejoint/quantum-circuits-23-17-local-gm-889-qubit-dqi),
  manwithacat's 888-qubit decoder HW-01 (https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook),
  Alex Chilton's 883-qubit decoder (based on jday96314's QuantumAI decoder and classical models), Ahmed Berat Özer,
  mrmeows, Dmitrii Gluzdov, ViberTeam, Phuc Thai (DePie).
- **dfthcsos** — Atakan Aldemir's learned angle words on SomeJoint's (Agniv Banerjee's team) R = 8, B = 143, C = 60
  factorization. The fifteen angle circuits, byte for byte, as assembled in Alex Chilton's portfolio (output of
  https://www.kaggle.com/code/alexchilton/22-57-lb-verified-three-task-portfolio, Apache-2.0; per-slot assembly credited
  there to manwithacat): slots 2, 3 and 14 from Dmitrii Gluzdov's
  https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source, slots 0, 1, 4, 5 and 8 ours (from
  our earlier versions), the rest from Atakan Aldemir's
  https://www.kaggle.com/code/atakanaldemir/qcc-888q-repair-and-native-chemistry-co-design (as packaged in
  https://www.kaggle.com/code/kirilltym/quantum-circuits-exact-gauge-5-word-qrom). Loader lineage as named there:
  Atakan Aldemir, heliosli, Dmitrii Gluzdov, on SomeJoint's design; factorization lineage: Michael Rustad, Ahmed Berat
  Özer, Akmal Xodarev (busyaprime), Vladislav Lavrentev (lightningv08), mrmeows. The table weights (W, H2_I, offset_W) are ours, starting from Ahmed Berat Özer's (output of
  https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression?scriptVersionId=363950507, Apache-2.0),
  with earlier work starting from manwithacat's (output of
  https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo, Apache-2.0, which builds on our earlier weights and
  on anazemcev's work), with earlier work starting from Dmitrii Gluzdov's (version 39 output of
  https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source, Apache-2.0), with
  earlier work starting from his versions 37 and 38, from Sargis Khachatryan's (avazaku; version 1 output
  of https://www.kaggle.com/code/avazaku/free-gap-sqp-889-qubit-dqi, Apache-2.0, built there on our earlier
  version) and from Ahmed Berat Özer's (v14 and v15 outputs of
  https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression, Apache-2.0) and using work from
  sigmaborov's (version 5 output of https://www.kaggle.com/code/sigmaborov/22-85569-lb-exact-gauge-search, Apache-2.0)
  and baidalinadilzhan's (https://www.kaggle.com/code/baidalinadilzhan/quantumai-improved-public-oct29, Apache-2.0;
  its code was used in making ours), all of which refine Atakan's.
  With thanks to Sargis Khachatryan (avazaku) for
  https://www.kaggle.com/code/avazaku/spectral-trust-epigraph-889-qubit-dqi.
- **Organizers** — task definitions, verifiers and kickmix (Google Quantum AI / Qualtran, competition data).